# SIGLENT SDS814X HD — network demo

Use the workspace **`.venv` kernel**. This notebook uses
[SCPI-Instrument-Control](https://github.com/little-did-I-know/SCPI-Instrument-Control)
7.3.0 over Ethernet at `131.215.200.196:5025`. That version is installed in the workspace.

**Run All reads the scope's existing stopped record**, plots it, and prints statistics.
First acquire the desired trace on the front panel and press **Stop**. The default channels
are **1, 2, 4**, which were enabled during setup; these numbers do not assign physical signals.
Channel scales, coupling, probes, and timebase remain as configured on the scope.
Reading waveforms changes the scope's transfer source/width/interval. No PCB or laser commands
are sent. Saving is off by default; connections close even if a transfer fails.

The initial bench check found a 10-million-point, 0.1-second record. Such a record is useful
for checking the connection, but is too short for the noise notebook's 0.1–10 Hz analysis.
For that experiment, acquire tens of seconds or longer, retain DC drive levels, and record
probe-floor measurements with the same settings.

In [ ]:
# Only needed in a new environment; uncomment and run, then restart the kernel.
# %pip install SCPI-Instrument-Control==7.3.0

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scpi_control import Oscilloscope

HOST, PORT = "131.215.200.196", 5025
CHANNELS = (1, 2, 4)
STOP_BEFORE_READ = False  # True explicitly stops a running acquisition; leaves it stopped.
SAVE = False
EXPORT_NOISE_CSV = False
# Assign actual probe connections before enabling the noise export.
SIGNAL_CHANNELS = {"fvoa1_mv": None, "fvoa2_mv": None, "pd_mv": None}
SETUP_NOTES = ""  # Probe locations, attenuation, bandwidth, instrument-floor measurement.

TOOLS = next(p.resolve() for p in (Path.cwd(), Path.cwd() / "tools", Path.cwd() / "hispec-tib/tools")
             if (p / "hispec_fibpcb.py").is_file())
DATA_DIR = TOOLS / "atten_noise_data"
print("SCPI-Instrument-Control", version("SCPI-Instrument-Control"))

## Connect and download one frozen record

All selected channels must already be enabled. WORD transfer preserves the scope's high-resolution
samples; the package handles binary framing, probe scaling, and the waveform time axis.
The same stopped acquisition supplies every channel. Do not change the front-panel settings
or resume acquisition during download. The scope stays stopped afterwards.

The package may log that SDS814X HD is not explicitly registered. Its family fallback selected
`modern`, four channels, and 100 MHz during the live check; the notebook checks the dialect below.

In [ ]:
waveforms = {}  # Clear previous results before each attempt.
if not CHANNELS or len(set(CHANNELS)) != len(CHANNELS) or any(c not in (1, 2, 3, 4) for c in CHANNELS):
    raise ValueError("Select distinct channel numbers from 1–4.")
with Oscilloscope(HOST, port=PORT, timeout=30.0) as scope:
    identity = dict(scope.device_info)
    display(identity)
    if identity["model"] != "SDS814X HD" or scope.dialect != "modern":
        raise RuntimeError("This demo expects the SDS814X HD using the modern command dialect.")
    settings = {c: getattr(scope, f"channel{c}").get_configuration() for c in CHANNELS}
    display(pd.DataFrame(settings.values()).set_index("channel"))
    if any(not cfg["enabled"] for cfg in settings.values()):
        raise RuntimeError("Enable the selected channels and acquire a fresh record on the scope first.")
    if STOP_BEFORE_READ:
        scope.stop()
    if scope.trigger.mode != "STOP":
        raise RuntimeError("Press Stop on the scope, or explicitly set STOP_BEFORE_READ=True.")
    captured_at = datetime.now(timezone.utc)
    pending = {c: scope.waveform.acquire(c, format="WORD", stride=1) for c in CHANNELS}
    if scope.trigger.mode != "STOP":
        raise RuntimeError("Acquisition resumed during download; discard these traces and repeat.")
    reference_time = pending[CHANNELS[0]].time
    for c, wf in pending.items():
        if len(wf.time) < 2 or not np.all(np.isfinite(wf.time)) or not np.all(np.diff(wf.time) > 0):
            raise ValueError(f"C{c}: invalid time axis.")
        if wf.voltage.shape != reference_time.shape or not np.array_equal(wf.time, reference_time):
            raise ValueError(f"C{c}: channels do not share an identical time axis.")
        if not np.all(np.isfinite(wf.voltage)):
            raise ValueError(f"C{c}: nonfinite voltage samples.")
    waveforms = pending  # Publish only a complete, checked set.
print(f"Downloaded {len(waveforms)} channels; connection closed; scope remains stopped.")

## Plot and inspect

Voltages are probe-referred volts from the library; the plots and table use millivolts.
`AC RMS` below is the standard deviation after removing the mean, over the full captured bandwidth.
It includes the scope/probe floor and is not a measurement restricted to the 0.1–10 Hz band.
Plots show a reduced selection of points for responsiveness; statistics and saved data use every point.
This display thinning is **not** an antialias filter or suitable input for noise spectra.

In [ ]:
rows = []
fig, axes = plt.subplots(len(waveforms), 1, sharex=True, figsize=(11, 2.6 * len(waveforms)), squeeze=False)
for ax, (c, wf) in zip(axes[:, 0], waveforms.items()):
    step = max(1, int(np.ceil(len(wf.time) / 20000)))
    ax.plot(wf.time[::step], wf.voltage[::step] * 1000, linewidth=0.6)
    ax.set(ylabel=f"C{c} (mV)")
    ax.grid(alpha=0.25)
    rows.append({"channel": c, "samples": len(wf.time), "duration_s": wf.time[-1] - wf.time[0],
                 "sample_rate_hz": wf.sample_rate, "mean_mv": np.mean(wf.voltage) * 1000,
                 "ac_rms_mv": np.std(wf.voltage) * 1000, "peak_to_peak_mv": np.ptp(wf.voltage) * 1000})
axes[-1, 0].set_xlabel("Scope time (s)")
fig.suptitle("Stored acquisition — display points thinned; full record retained")
fig.tight_layout()
plt.show()
display(pd.DataFrame(rows).set_index("channel"))

## Optional archive and noise-notebook export

Set `SAVE=True` to save each full waveform with the package's provenance-bearing NPZ format,
plus setup metadata. Files go into `tools/atten_noise_data/` under a unique UTC capture name.
To load an archive later: `from scpi_control.waveform_io import load_waveform` (see the package documentation).

For the final section of `atten_and_tput_cal_and_noise_lab.ipynb`, assign `SIGNAL_CHANNELS`,
enter `SETUP_NOTES`, and set both `SAVE=True` and `EXPORT_NOISE_CSV=True`. The resulting CSV has
`t_s,fvoa1_mv,fvoa2_mv,pd_mv`; copy its path into that notebook's `SCOPE_CSV`.
FVOA channels must probe **post-amplifier drive voltages**, DC coupled. The PD column retains the
voltage at the probed node; set `SCOPE_PD_TO_ADC` in the analysis notebook to account for its divider.
This scope clock is shared across the three traces, but is not synchronized with PCB timestamps.

A full-record CSV can be large. For low-frequency work choose suitable scope acquisition settings;
use an antialias filter before any later numerical decimation, not the plot's point selection.

In [ ]:
if SAVE:
    if EXPORT_NOISE_CSV:
        channels = list(SIGNAL_CHANNELS.values())
        if any(c not in waveforms for c in channels) or len(set(channels)) != 3:
            raise ValueError("Assign three distinct captured channels to SIGNAL_CHANNELS.")
        if not SETUP_NOTES.strip():
            raise ValueError("Record the probe locations and setup in SETUP_NOTES.")
        if any(settings[SIGNAL_CHANNELS[name]]["coupling"] != "DC" for name in ("fvoa1_mv", "fvoa2_mv")):
            raise ValueError("The downstream slope calculation needs DC-coupled FVOA means.")
    DATA_DIR.mkdir(exist_ok=True)
    stem = "siglent_" + captured_at.strftime("%Y%m%dT%H%M%S_%fZ")
    # Waveform saving is local file I/O; the scope session above is already closed.
    for c, wf in waveforms.items():
        path = DATA_DIR / f"{stem}_c{c}.npz"
        scope.waveform.save_waveform(wf, str(path))
        print(path)
    metadata = {"instrument": identity, "captured_at_utc": captured_at.isoformat(),
                "package_version": version("SCPI-Instrument-Control"), "settings": settings,
                "signal_channels": SIGNAL_CHANNELS, "notes": SETUP_NOTES,
                "transfer_format": "WORD", "transfer_stride": 1}
    (DATA_DIR / f"{stem}.json").write_text(json.dumps(metadata, indent=2) + "\n")
    if EXPORT_NOISE_CSV:
        csv_path = DATA_DIR / f"{stem}_noise.csv"
        columns = {"t_s": reference_time}
        columns.update({name: waveforms[c].voltage * 1000 for name, c in SIGNAL_CHANNELS.items()})
        pd.DataFrame(columns).to_csv(csv_path, index=False)
        print("SCOPE_CSV =", repr(str(csv_path)))
else:
    print("Nothing saved. Set SAVE=True above and rerun this cell to archive these traces.")

## Acquire another record

Use **Run/Stop on the scope**, then rerun the download and plot cells. This preserves your chosen
trigger, scale, coupling, and timebase settings. Channels can be changed with `CHANNELS` before download.
For a programmatic resume, the optional cell below calls the package's `run()` method; it starts
acquisition and closes the connection. Stop again before downloading the next record.

In [ ]:
RESUME_ACQUISITION = False
if RESUME_ACQUISITION:
    with Oscilloscope(HOST, port=PORT, timeout=10.0) as scope:
        scope.run()
    print("Scope acquisition resumed.")